# Fraud Review Agent with MongoDB Atlas and Claude Managed Agents

Build a human-in-the-loop fraud-review agent that uses MongoDB Atlas as its retrieval engine, graph store, and system of record -- driven by Claude Managed Agents (CMA) custom tools.

| # | Section | Description |
|---|---------|-------------|
| [1](#1-connect-mongodb-to-a-managed-agent) | **Connect MongoDB to a managed agent** | Three credential-safe data paths (Path A recommended) |
| [2](#2-retrieve-four-patterns-one-engine) | **Retrieve: four patterns, one engine** | Vector, full-text, hybrid RRF, and graph pipeline builders |
| [3](#3-the-end-to-end-agent-human-in-the-loop-fraud-review) | **End-to-end agent: human-in-the-loop fraud review** | AP2 mandates, custom tools, escalation gate loop |
| [4](#4-mongodb-atlas-as-the-system-of-record-and-audit-backbone) | **System of record and audit backbone** | Decisions, audit trail, and mandate receipts in one cluster |
| [5](#5-cheat-sheet) | **Cheat sheet** | Pipeline signatures, tool schemas, config constants |
| [6](#6-recap) | **Recap** | Connect, retrieve, gate, persist |

### Why MongoDB Atlas?

Most agent stacks bolt together three or four systems: a vector database for semantic search, a separate engine for keywords, a graph store for relationships, and an operational database for the records themselves. Every seam is another integration, another credential, another place the agent's view of the world can drift.

**MongoDB collapses that into one engine.** The same documents are searchable by **vector** (`$vectorSearch`), by **full-text** (`$search`), as a **hybrid** of the two fused with reciprocal rank fusion (`$rankFusion`), and traversable as a **graph** (`$graphLookup`) -- and they are the same documents the agent reads, writes, and persists its decisions to. One query language, one cluster, one connection (`pymongo`).

**Where the Agent Adds Value:** CMA runs the agent loop on Anthropic's side. Your process owns the data path: the agent reasons over MongoDB retrieval, pauses for humans on risky calls via `requires_action`, and writes verdicts back -- without the DB credential ever entering the agent context.

> MongoDB here is your **operational data store and retrieval engine** -- your system of record, controlled by your application. That's distinct from the memory features CMA manages natively on the platform. The two are complementary.

**By the end you will be able to:**

- Connect a credential-safe MongoDB Atlas data path into a Claude Managed Agent in three ways.
- Lift vector, full-text, hybrid, and graph retrieval into an agent's custom tools.
- Gate risky agent decisions behind CMA's native human-in-the-loop pause.
- Make one MongoDB Atlas cluster the agent's system of record and audit backbone.

In [1]:
%%capture
# Third-party dependencies: the Anthropic SDK (drives Claude Managed Agents via client.beta.*),
# pymongo (the host-side data path), python-dotenv, and cryptography + pyjwt (sign/verify the
# AP2 mandates). This installs the PyPI packages only — the notebook also imports the adjacent
# `mongodb_on_cma/` package and `utilities.py`, so run it from a clone of this repo (see Setup).
%pip install -q "anthropic>=0.109.0" pymongo python-dotenv cryptography pyjwt

## Prerequisites

**Required knowledge:** Python, `pymongo` basics, and the Claude API. New to MongoDB + Claude? Start with the [library-RAG notebook](../third_party/MongoDB/rag_using_mongodb.ipynb). New to the custom-tool gate? See [`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb).

**Required tools:** Python 3.11+, an [Anthropic API key](https://console.anthropic.com), and a MongoDB Atlas cluster (free M0 works).

## Setup

```bash
uv sync --all-extras   # from the repo root
cp .env.example .env    # then add MONGO_URI and Anthropic auth
```

### Environment variables

| Variable | Required | Description |
| --- | --- | --- |
| `MONGO_URI` | Yes | Atlas SRV connection string. A **free M0 cluster runs everything** in this cookbook. |
| `ANTHROPIC_API_KEY` | Yes | Or use `ANTHROPIC_AUTH_TOKEN` / `ANTHROPIC_PROFILE` (e.g. `ant` CLI federation). |
| `MDB_ATLAS_API_KEY` | No | MongoDB Atlas AI endpoint for embeddings + reranking. Alternative to Voyage. |
| `VOYAGE_API_KEY` | No | Voyage AI SDK for embeddings. Alternative to Atlas AI. Seed ships precomputed vectors, so **neither provider key is needed** to run the cookbook. |
| `ENABLE_RERANK` | No | Set to `1` to add a reranker second stage to hybrid search. Requires a provider key above. |
| `AUTO_APPROVE` | No | Set to `1` to resolve the human gate deterministically (for CI / non-interactive runs). Overrides the agent on the seeded structuring case so a differing human verdict is visible. |
| `COOKBOOK_MODEL` | No | Override the agent model. Default: `claude-haiku-4-5`. |

**No Atlas cluster yet?** The partner notebook [`rag_using_mongodb.ipynb`](../third_party/MongoDB/rag_using_mongodb.ipynb) walks through creating a free cluster and getting your `MONGO_URI`; the [Atlas Search index docs](https://www.mongodb.com/docs/atlas/atlas-search/create-index/) cover index setup.

The retrieval builders and gate loop are defined **inline** in this notebook. The [`mongodb_on_cma/`](mongodb_on_cma/) package holds setup boilerplate: [`config.py`](mongodb_on_cma/config.py), [`embeddings.py`](mongodb_on_cma/embeddings.py), [`tools.py`](mongodb_on_cma/tools.py), [`ap2_mandates.py`](mongodb_on_cma/ap2_mandates.py), and [`seed.py`](mongodb_on_cma/seed.py).

In [2]:
import hashlib
import json
import logging
import os
from datetime import UTC, datetime
from typing import Any

import dotenv
from anthropic import Anthropic
from mongodb_on_cma import (
    EMBED_DIM,
    build_audit_event,
    build_decision_doc,
    ensure_indexes,
    has_anthropic_auth,
    load_seed,
    make_embedding_client,
    missing_required_env,
    preflight,
    prepare_seed,
    rerank,
    resolve_model,
    seed_collection,
    server_version,
    supports_rank_fusion,
)
from mongodb_on_cma.ap2_mandates import (
    attach_mandates,
    store_mandate_receipt,
    tool_verify_mandates,
)
from pymongo import MongoClient
from utilities import wait_for_idle_status

dotenv.load_dotenv()

missing = missing_required_env()
assert not missing, f"Set these in your environment / .env: {missing}"
# Anthropic auth is a soft check: an API key, an auth token, or a profile (e.g. the `ant`
# CLI's workload-identity federation) all work, so warn rather than block if none is set.
if not has_anthropic_auth():
    print(
        "warning: no ANTHROPIC_API_KEY / ANTHROPIC_AUTH_TOKEN / ANTHROPIC_PROFILE found; "
        "relying on the SDK / `ant` CLI to resolve credentials."
    )

MODEL = resolve_model()
ENABLE_RERANK = os.getenv("ENABLE_RERANK", "").lower() in ("1", "true")
AUTO_APPROVE = os.getenv("AUTO_APPROVE", "").lower() in ("1", "true")

# Quiet the SDK's one-shot notice that ANTHROPIC_API_KEY shadows profile/federation
# auto-discovery — expected when an API key is set; harmless under `ant` / WIF auth.
logging.getLogger("anthropic.lib.credentials._auth").setLevel(logging.ERROR)
client = Anthropic()
mongo = MongoClient(os.environ["MONGO_URI"], appname="partner-cookbook-python-cma")
db = mongo["fraud_review_demo"]
coll = db["transactions"]
ai_client = make_embedding_client()  # Atlas / Voyage / None (seed ships precomputed vectors)

print(
    f"model={MODEL}  rerank={'on' if ENABLE_RERANK else 'off'}  provider={'yes' if ai_client else 'none'}"
)

model=claude-haiku-4-5  rerank=off  provider=none


## Architecture

```mermaid
graph LR
    subgraph YourProcess["Your Process (host-side)"]
        ENV[".env\nMONGO_URI\nANTHROPIC_API_KEY"]
        PYMONGO["pymongo"]
        HANDLERS["Tool Handlers\nverify_mandates\nget_transaction\nhybrid_search\ndetect_fraud_ring\nrecord_decision"]
    end

    subgraph Anthropic["Anthropic (cloud)"]
        AGENT["CMA Agent Loop\nclient.beta.sessions"]
        SANDBOX["Sandbox\n(no DB credential)"]
    end

    subgraph Atlas["MongoDB Atlas"]
        TXN["transactions\n(operational)"]
        DECISIONS["transaction_decisions\n(append-only)"]
        AUDIT["audit_events\n(append-only)"]
        MANDATES["mandate_receipts"]
        VIDX["Vector Index\n($vectorSearch)"]
        SIDX["Search Index\n($search, $rankFusion)"]
    end

    AGENT -- "custom_tool_use\n(session idles)" --> HANDLERS
    HANDLERS -- "custom_tool_result\n(session resumes)" --> AGENT
    HANDLERS --> PYMONGO
    PYMONGO --> Atlas
    ENV -.-> PYMONGO

    style YourProcess fill:#e8f4e8,stroke:#2d6a2d
    style Anthropic fill:#e8e8f4,stroke:#2d2d6a
    style Atlas fill:#f4e8e8,stroke:#6a2d2d
```

## 1. Connect MongoDB to a managed agent

The agent loop is Anthropic-hosted. You host only the **data path** to MongoDB -- the credential never enters the agent context or sandbox.

> **CMA vs. Claude Agent SDK:** This cookbook uses the CMA beta sessions API (`client.beta.sessions`), which runs the agent loop on Anthropic's infrastructure. The [Claude Agent SDK](https://code.claude.com/docs/en/agent-sdk/overview) is a separate product that runs agents in your own process. The two are complementary; the CMA approach is what keeps the credential boundary intact.

> **The credential never leaves your process.** Path A (host-side custom tool) means `MONGO_URI` stays in your `.env` and your handler functions. The CMA agent session idles with `requires_action`, your process runs the `pymongo` query, and you post back the result. The sandbox never sees the connection string.

### Path A: host-side custom tool (recommended)

Define a `custom` tool. On each call the session pauses (`requires_action`), your process runs the `pymongo` query, and you post back a `custom_tool_result`. Here is the full round-trip with a minimal `find_notes` tool:

```mermaid
sequenceDiagram
    participant User as Your Process
    participant CMA as CMA Agent (Anthropic)
    participant MongoDB as MongoDB Atlas

    User->>CMA: user.message ("Review txn-review-clean")
    activate CMA
    CMA->>CMA: Agent reasons, picks tool

    CMA-->>User: agent.custom_tool_use (get_transaction)
    Note over CMA: Session idles<br/>stop_reason: requires_action
    deactivate CMA

    User->>MongoDB: pymongo find_one({transaction_id})
    MongoDB-->>User: transaction document

    User->>CMA: user.custom_tool_result (JSON)
    activate CMA
    CMA->>CMA: Agent reasons with result

    CMA-->>User: agent.custom_tool_use (hybrid_search_similar_frauds)
    Note over CMA: Session idles again
    deactivate CMA

    User->>MongoDB: pymongo aggregate($rankFusion pipeline)
    MongoDB-->>User: similar precedents

    User->>CMA: user.custom_tool_result (JSON)
    activate CMA
    CMA->>CMA: Agent decides

    CMA-->>User: agent.custom_tool_use (record_decision)
    deactivate CMA

    User->>MongoDB: pymongo insert_one (decision + audit)
    MongoDB-->>User: ack

    User->>CMA: user.custom_tool_result
    activate CMA
    Note over CMA: Session idles<br/>stop_reason: end_turn
    deactivate CMA
```

In [3]:
notes = mongo["cookbook_mongodb_on_cma"]["notes"]
notes.delete_many({})
notes.insert_many(
    [
        {"note_id": "n1", "tag": "todo", "text": "Renew the SSL certificate before it expires."},
        {"note_id": "n2", "tag": "idea", "text": "Add a dark-mode toggle to the dashboard."},
        {"note_id": "n3", "tag": "todo", "text": "Email the quarterly report to the finance team."},
    ]
)

notes_agent = client.beta.agents.create(
    name="cookbook-mongodb-notes",
    model=MODEL,
    system=(
        "You answer questions about the user's notes. Use the find_notes tool to fetch notes "
        "(optionally filtered by `tag`) before answering, then give a concise final answer."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": True, "permission_policy": {"type": "always_allow"}},
        },
        {
            "type": "custom",
            "name": "find_notes",
            "description": "Return stored notes, optionally filtered by tag (e.g. 'todo', 'idea').",
            "input_schema": {
                "type": "object",
                "properties": {"tag": {"type": "string", "description": "optional tag filter"}},
                "required": [],
            },
        },
    ],
)
notes_env = client.beta.environments.create(
    name="cookbook-mongodb-notes-env", config={"type": "cloud", "networking": {"type": "limited"}}
)
notes_session = client.beta.sessions.create(
    environment_id=notes_env.id,
    agent={"type": "agent", "id": notes_agent.id, "version": notes_agent.version},
    title="MongoDB notes (host-side custom tool)",
)
print(f"seeded {notes.count_documents({})} notes; session: {notes_session.id}")

seeded 3 notes; session: sesn_01BuNiXnZupYkiQNsnr2XEfH


Stream events, handle each `agent.custom_tool_use` by running `pymongo` host-side, and post back the result. Project with `{"_id": 0}` since `ObjectId` is not JSON-serializable.

In [4]:
def run_find_notes(args: dict) -> dict:
    """Host-side handler: pymongo runs here; _id (ObjectId) is projected out so the result is JSON-able."""
    flt = {"tag": args["tag"]} if args.get("tag") else {}
    return {"notes": list(notes.find(flt, {"_id": 0}))}


tool_use_events, responded, round_trips = {}, set(), []
question = "Using find_notes, list every note tagged 'todo', then tell me how many there are."

with client.beta.sessions.events.stream(notes_session.id) as stream:
    client.beta.sessions.events.send(
        session_id=notes_session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
    )
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded:
                        continue
                    tev = tool_use_events[event_id]
                    result = (
                        run_find_notes(tev.input)
                        if tev.name == "find_notes"
                        else {"error": "unknown"}
                    )
                    if tev.name == "find_notes":
                        round_trips.append(
                            {"tool_input": tev.input, "returned": len(result["notes"])}
                        )
                    client.beta.sessions.events.send(
                        session_id=notes_session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, notes_session.id)
print("host-side find_notes round-trips:")
for rt in round_trips:
    print(f"  find_notes({rt['tool_input']}) -> {rt['returned']} notes")

client.beta.sessions.archive(notes_session.id)
client.beta.environments.archive(notes_env.id)
client.beta.agents.archive(notes_agent.id)
notes.drop()
print("archived the demo agent / environment / session; dropped the notes collection")

host-side find_notes round-trips:
  find_notes({'tag': 'todo'}) -> 2 notes
archived the demo agent / environment / session; dropped the notes collection


### Path comparison

| | **Path A: Host-side custom tool** | **Path B: In-sandbox pymongo** | **Path C: MCP server** |
|---|---|---|---|
| **Where credential lives** | Your process only (never enters sandbox or agent context) | Sandbox container env var | Your MCP server (vault-backed) |
| **Query flexibility** | Fixed, audited set of queries you define | Full pymongo surface (agent writes arbitrary queries) | Full MCP surface (find, aggregate, $vectorSearch) |
| **Setup complexity** | Lowest -- no standing server, just handler functions | Requires self-hosted sandbox with pymongo installed | Requires hosting the MongoDB MCP server behind HTTPS |
| **Recommended for** | Most use cases; secret-bearing data sources; production | Self-hosted sandboxes where direct DB access is acceptable | Broad query surface without fixed tool schemas |

Path A is the default for this cookbook. When in doubt, stay on Path A.

## 2. Retrieve: four patterns, one engine

> **One engine, four retrieval patterns.** Vector, full-text, hybrid RRF, and graph traversal all run against the same MongoDB documents in the same cluster. No separate vector database, no external search engine, no graph store -- one `pymongo` connection, one query language, one set of documents.

Four retrieval patterns, each a plain aggregation-pipeline builder. Section 3's agent calls these same functions through its custom tools.

### Pattern flowcharts

#### Vector Search
```mermaid
flowchart LR
    Q["Query text"] --> E["Embed\n(Voyage / Atlas AI)"]
    E --> VS["$vectorSearch\nindex: vector_index\npath: embedding\nnumCandidates: limit*10"]
    VS --> F["Filter\nstatus in decided"]
    F --> P["$project\nfields + score"]
    P --> R["Top-k results"]
```

#### Full-Text Search
```mermaid
flowchart LR
    Q["Query string"] --> S["$search\nindex: search_index\ntext.query + text.path"]
    S --> L["$limit k"]
    L --> P["$project\nfields"]
    P --> R["Top-k results"]
```

#### Hybrid (Reciprocal Rank Fusion)
```mermaid
flowchart LR
    Q["Query text +\nquery vector"]
    Q --> RF["$rankFusion"]
    subgraph RF_inner["$rankFusion input pipelines"]
        V["vector branch\n$vectorSearch"]
        L["lexical branch\n$search + $limit"]
    end
    RF --> RF_inner
    RF_inner --> FUSE["RRF score fusion\n(server-side)"]
    FUSE --> LIM["$limit k"]
    LIM --> P["$project\nfields + score"]
```

#### Graph Traversal
```mermaid
flowchart LR
    A["Seed account_id"] --> M["$match\nsender.account_number"]
    M --> G["$graphLookup\nfrom: transactions\nconnectFrom: recipient.account_number\nconnectTo: sender.account_number\nmaxDepth: 4"]
    G --> S["summarize_ring()\ncircular_flow?\nlayering?\nnetwork_size"]
    S --> R["Ring signals"]
```

### Comparison

| Pattern | MongoDB Stage | Best For | Catches | Misses |
|---------|--------------|----------|---------|--------|
| **Vector** | `$vectorSearch` | Semantic similarity, fuzzy matches | Conceptually similar cases regardless of wording | Exact names, IDs, codes |
| **Full-text** | `$search` (BM25) | Keyword precision, exact terms | Specific names, account numbers, amounts | Paraphrases, conceptual similarity |
| **Hybrid (RRF)** | `$rankFusion` | Best of both -- production default | Semantic and keyword matches, fused server-side | Relationship/network signals |
| **Graph** | `$graphLookup` | Network traversal, fraud rings | Mule chains, layering, circular money flow | Content similarity |

Load the seed fixture and build indexes, then demo each pattern.

In [5]:
# The four retrieval builders — each returns a plain aggregation pipeline you can lift into your
# own collection. These are the functions Section 3's agent calls through its custom tools.
from mongodb_on_cma.config import (
    DECIDED_STATUSES,
    LEXICAL_PATHS,
    PROJECT_FIELDS,
    SEARCH_INDEX_NAME,
    VECTOR_INDEX_NAME,
)


def _project_stage(with_score: bool = False) -> dict:
    proj: dict[str, Any] = {f: 1 for f in PROJECT_FIELDS}
    proj["_id"] = 0
    if with_score:
        proj["score"] = {"$meta": "score"}
    return {"$project": proj}


def build_vector_pipeline(
    qvec, *, limit, candidates=None, vector_index=VECTOR_INDEX_NAME, status_in=DECIDED_STATUSES
) -> list[dict]:
    return [
        {
            "$vectorSearch": {
                "index": vector_index,
                "path": "embedding",
                "queryVector": qvec,
                "numCandidates": candidates or max(50, limit * 10),
                "limit": limit,
                "filter": {"status": {"$in": list(status_in)}},
            }
        },
        _project_stage(),
    ]


def build_lexical_pipeline(query, *, limit, search_index=SEARCH_INDEX_NAME) -> list[dict]:
    return [
        {"$search": {"index": search_index, "text": {"query": query, "path": LEXICAL_PATHS}}},
        {"$limit": limit},
        _project_stage(),
    ]


def build_rank_fusion_pipeline(
    qvec,
    query,
    *,
    k,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    status_in=DECIDED_STATUSES,
) -> list[dict]:
    # $rankFusion (MongoDB 8.0+) runs both input pipelines and fuses them by reciprocal rank —
    # one aggregation, server-side. Each input pipeline gets weight 1 (uniform). To bias toward
    # semantic vs. lexical, add "combination": {"weights": {"vector": w, "lexical": w}}.
    candidates = max(50, k * 10)
    per_branch = max(k * 4, 20)
    return [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vector": [
                            {
                                "$vectorSearch": {
                                    "index": vector_index,
                                    "path": "embedding",
                                    "queryVector": qvec,
                                    "numCandidates": candidates,
                                    "limit": per_branch,
                                    "filter": {"status": {"$in": list(status_in)}},
                                }
                            }
                        ],
                        "lexical": [
                            {
                                "$search": {
                                    "index": search_index,
                                    "text": {"query": query, "path": LEXICAL_PATHS},
                                }
                            },
                            {"$limit": per_branch},
                        ],
                    }
                },
            }
        },
        {"$limit": k},
        _project_stage(with_score=True),
    ]


def build_graph_pipeline(
    account_id: str, *, max_depth: int = 4, collection: str = "transactions"
) -> list[dict]:
    return [
        {"$match": {"sender.account_number": account_id}},
        {
            "$graphLookup": {
                "from": collection,
                "startWith": "$recipient.account_number",
                "connectFromField": "recipient.account_number",
                "connectToField": "sender.account_number",
                "as": "chain",
                "maxDepth": max_depth,
                "depthField": "depth",
            }
        },
    ]


def summarize_ring(graph_doc: dict, *, seed_account: str) -> dict:
    # Turn a $graphLookup chain into fraud-ring signals: circular flow back to the seed account,
    # layering (many small transfers), and overall network size.
    chain: list[dict] = list(graph_doc.get("chain", []))
    accounts: set[str] = set()
    small_transfers = 0
    circular_flow = False
    for edge in chain:
        sender = (edge.get("sender") or {}).get("account_number")
        recipient = (edge.get("recipient") or {}).get("account_number")
        accounts.update(a for a in (sender, recipient) if a)
        if recipient == seed_account:
            circular_flow = True
        if float(edge.get("amount", 0) or 0) < 1000:
            small_transfers += 1
    network_size = len(chain)
    layering = small_transfers >= 5
    return {
        "network_size": network_size,
        "unique_accounts": len(accounts),
        "circular_flow": circular_flow,
        "layering": layering,
        "suspicious_patterns": circular_flow or layering or network_size >= 3,
    }

In [6]:
docs = prepare_seed(load_seed(), now=datetime.now(UTC))
count = seed_collection(coll, docs)
print(f"seeded {count} transactions")

# Create the vector + Atlas Search indexes if absent, then block until both are queryable AND
# actually reflect the freshly-seeded docs (Atlas Search is eventually consistent).
ensure_indexes(coll, dim=EMBED_DIM)

check = preflight(coll)
for issue in check["issues"]:
    print("PREFLIGHT:", issue)
assert check["ok"], "Fix the preflight issues above before continuing."

# This cookbook uses the native `$rankFusion` stage for hybrid search, which needs MongoDB 8.0+.
# Every current Atlas cluster — including the free M0 tier — runs 8.0 or later, so this holds by
# default; the check just fails loudly if you point MONGO_URI at an older self-hosted server.
version = server_version(coll)
assert supports_rank_fusion(version), (
    f"MongoDB {version} predates $rankFusion (needs 8.0+). Use an Atlas cluster (any tier) or a "
    f"self-hosted MongoDB 8.0+."
)
print(f"server={version}  hybrid search=$rankFusion (native)")

seeded 20 transactions
server=8.0.28  hybrid search=$rankFusion (native)


### Pattern 1: Vector search (`$vectorSearch`)

Find documents whose embedding is nearest the query vector. Uses an existing document's embedding as the query -- no embedding API call needed.

In [7]:
query_doc = coll.find_one({"transaction_id": "txn-review-struct"})  # a pending case
hits = list(coll.aggregate(build_vector_pipeline(query_doc["embedding"], limit=3)))
print("nearest decided precedents:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

nearest decided precedents:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### Pattern 2: Full-text search (`$search`, BM25)

Keyword and phrase matching -- catches exact names, IDs, and codes that embeddings blur.

In [8]:
hits = list(
    coll.aggregate(
        build_lexical_pipeline("cash deposit just under the reporting threshold", limit=3)
    )
)
print("full-text matches:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

full-text matches:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-review-struct: Cash deposit of 4950 USD, just under the 5000 reporting threshold, follo
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account


### Pattern 3: Hybrid (reciprocal rank fusion)

`$rankFusion` (MongoDB 8.0+) fuses vector and full-text rankings server-side in one aggregation -- no client-side merging.

In [9]:
qvec, query = query_doc["embedding"], "structuring: cash deposit just under the threshold"
hits = list(coll.aggregate(build_rank_fusion_pipeline(qvec, query, k=3)))
print("hybrid via $rankFusion (server-side):")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

hybrid via $rankFusion (server-side):
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### Pattern 4: Graph traversal (`$graphLookup`)

Follow `sender -> recipient` links to surface circular money-flow rings. A relationship signal, orthogonal to similarity ranking.

In [10]:
graph_doc = next(
    iter(coll.aggregate(build_graph_pipeline("ACC-RING-A", collection=coll.name))), {"chain": []}
)
ring = summarize_ring(graph_doc, seed_account="ACC-RING-A")
print(
    f"graph traversal from ACC-RING-A: network_size={ring['network_size']} "
    f"circular_flow={ring['circular_flow']} suspicious={ring['suspicious_patterns']}"
)

graph traversal from ACC-RING-A: network_size=4 circular_flow=True suspicious=True


Each builder is a generic shape -- swap the embedding field, searchable paths, fusion weights, or graph edges to fit your domain. Lift them straight into your own project.

### MongoDB Atlas Reference

| Feature | Documentation |
|---------|--------------|
| `$vectorSearch` | [Atlas Vector Search Stage](https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/) |
| `$search` | [Atlas Search Text Query](https://www.mongodb.com/docs/atlas/atlas-search/query/text/) |
| `$rankFusion` | [Reciprocal Rank Fusion](https://www.mongodb.com/docs/manual/reference/operator/aggregation/rankFusion/) |
| `$graphLookup` | [Graph Lookup Aggregation](https://www.mongodb.com/docs/manual/reference/operator/aggregation/graphLookup/) |
| Atlas Search indexes | [Create an Atlas Search Index](https://www.mongodb.com/docs/atlas/atlas-search/create-index/) |
| pymongo aggregation | [PyMongo Aggregation Examples](https://pymongo.readthedocs.io/en/stable/examples/aggregation.html) |

## 3. The end-to-end agent: human-in-the-loop fraud review

> **Append-only audit trail in the same cluster.** Every decision writes to `transaction_decisions` (immutable verdict) and `audit_events` (append-only trail) alongside the operational `transactions` collection. Escalations record both the agent's recommendation and the human's final call. One cluster is the system of record *and* the compliance surface.

The agent reviews flagged transactions using the Section 2 retrieval patterns as custom tools, pauses for a human on risky cases, and writes decisions back to the same cluster.

<p align="center">
  <img src="../images/cma_with_mongodb_atlas.jpg" width="70%" alt="Agent loop on Anthropic; data path in your notebook." />
</p>

```mermaid
flowchart TD
    START["Receive pending\ntransaction IDs"] --> NEXT["Pick next transaction"]
    NEXT --> VM["verify_mandates\n(AP2 signature, constraints,\ndouble-spend)"]

    VM -- "valid=false OR\nconstraints_satisfied=false OR\ndouble_spend_detected=true" --> REJECT_HARD["record_decision\ndecision=reject\n(hard gate)"]

    VM -- "mandates OK" --> GT["get_transaction"]
    GT --> HS["hybrid_search_similar_frauds\n($rankFusion precedents)"]
    HS --> DR["detect_fraud_ring\n($graphLookup ring signals)"]

    DR --> DECIDE{Decision logic}

    DECIDE -- "High confidence +\nno flags" --> RD["record_decision\n(approve or reject)"]

    DECIDE -- "Structuring amount\n($4,900-$4,999)" --> ESC["escalate\n(requires_action)"]
    DECIDE -- "High value\n(>= $50,000) + approve" --> ESC
    DECIDE -- "Ring:\nsuspicious_patterns=true" --> ESC
    DECIDE -- "Medium confidence\n(~75-85%)" --> ESC

    ESC --> HUMAN["Human reviewer\n(session idles)"]
    HUMAN --> RD_ESC["record_decision\nescalated=true\nhuman's verdict"]

    REJECT_HARD --> MORE{More\ntransactions?}
    RD --> MORE
    RD_ESC --> MORE
    MORE -- yes --> NEXT
    MORE -- no --> DONE["Session ends"]

    style HUMAN fill:#fff3cd,stroke:#856404
    style REJECT_HARD fill:#f8d7da,stroke:#721c24
    style ESC fill:#fff3cd,stroke:#856404
```

### AP2 mandates: verify, then reason

Before behavioral analysis, the agent verifies an [AP2](https://github.com/google-agentic-commerce/AP2) mandate -- a cryptographically signed credential proving user authorization. Call `verify_mandates` and act on its verdict (valid, constraints-satisfied, double-spend). The signing details stay in [`ap2_mandates.py`](mongodb_on_cma/ap2_mandates.py).

The cell below generates a keypair and attaches signed mandates to pending cases. The private key stays in local Python state.

In [11]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec

ts_private_key = ec.generate_private_key(ec.SECP256R1())  # Trusted Surface key, fresh each run
AGENT_PK = (
    ts_private_key.public_key()
    .public_bytes(serialization.Encoding.X962, serialization.PublicFormat.UncompressedPoint)
    .hex()
)
ts_public_key = ts_private_key.public_key()

PENDING = [
    "txn-review-clean",
    "txn-review-fraud",
    "txn-review-struct",  # seeded human-override case under AUTO_APPROVE
    "txn-review-high",
    "txn-review-ring",
]
mandates = attach_mandates(coll, PENDING, agent_pk=AGENT_PK, ts_private_key=ts_private_key)
print(f"attached AP2 mandates to {len(mandates)} transactions  agent_pk={AGENT_PK[:16]}…")

attached AP2 mandates to 5 transactions  agent_pk=0406c4adc2db2092…


### Custom tools and host-side handlers

Each tool is a `type: "custom"` tool definition. When the agent calls one, the session pauses with `requires_action`, **this notebook** runs the handler with `pymongo`, and sends the JSON result back. `HANDLERS` maps the five data tools; `escalate` is deliberately absent because the gate loop routes it to the human resolver instead.

The tool implementations are split into three cells:
1. **Helpers + per-tool functions** -- the `pymongo` work behind each tool
2. **Tool schemas** -- the `input_schema` definitions the agent sees
3. **Handler dispatch** -- wires tool names to handler functions

Serialization helpers and rerank utilities (shared across tool handlers):

In [ ]:
# The host-side tool implementations — the pymongo work behind each custom tool. These run in
# THIS process (never the sandbox), reusing the Section 2 builders. The dispatch wiring + the
# AP2/record-decision wrappers are in the next cell; build_decision_doc / build_audit_event
# (imported above) shape the persisted documents and are shared with the AP2 module.


def _jsonable(value):
    """Recursively coerce Mongo/BSON values (ObjectId, datetime, ...) into JSON-safe types."""
    if isinstance(value, dict):
        return {k: _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def _pick_fields(candidate):
    keep = ("transaction_id", "text", "amount", "sender", "recipient", "decision", "score")
    return _jsonable({k: candidate[k] for k in keep if k in candidate})


def rerank_pool_size(k: int, fanout: int, *, enable_rerank: bool) -> int:
    return k * fanout if enable_rerank else k


def merge_rerank(candidates, rerank_results, *, top_k) -> list[dict]:
    def _score(r):
        s = getattr(r, "relevance_score", None)
        return (
            (s if s is not None else r.get("relevance_score", float("-inf")))
            if isinstance(r, dict)
            else (s if s is not None else float("-inf"))
        )

    seen: set[str] = set()
    out: list[dict] = []
    for result in sorted(rerank_results, key=_score, reverse=True):
        index = getattr(result, "index", None)
        if index is None and isinstance(result, dict):
            index = result.get("index")
        score = getattr(result, "relevance_score", None)
        if score is None and isinstance(result, dict):
            score = result.get("relevance_score")
        if index is None or index < 0 or index >= len(candidates):
            continue
        candidate = candidates[index]
        cid = str(candidate.get("transaction_id", index))
        if cid in seen:
            continue
        seen.add(cid)
        out.append({**candidate, "score": score})
        if len(out) >= top_k:
            break
    return out


The four tool functions -- each wraps a Section 2 pipeline builder with the `pymongo` call:

In [ ]:
def tool_get_transaction(coll, transaction_id) -> dict:
    doc = coll.find_one({"transaction_id": transaction_id})
    if not doc:
        return {"error": "not_found", "transaction_id": transaction_id}
    return _jsonable({k: v for k, v in doc.items() if k not in ("embedding", "_id")})


def tool_hybrid_search_similar_frauds(
    coll,
    transaction_id,
    k,
    *,
    enable_rerank=False,
    fanout=5,
    reranker=None,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    status_in=DECIDED_STATUSES,
) -> dict:
    txn = coll.find_one({"transaction_id": transaction_id})
    if not txn:
        return {"error": "not_found", "transaction_id": transaction_id, "similar": []}
    qvec = txn["embedding"]
    query = txn.get("text", "")
    pool_k = rerank_pool_size(k, fanout, enable_rerank=enable_rerank)
    candidates = list(
        coll.aggregate(
            build_rank_fusion_pipeline(
                qvec,
                query,
                k=pool_k,
                vector_index=vector_index,
                search_index=search_index,
                status_in=status_in,
            )
        )
    )
    candidates = [c for c in candidates if str(c.get("transaction_id")) != str(transaction_id)]
    if enable_rerank and reranker is not None and candidates:
        results = reranker(query, [c.get("text", "") for c in candidates], top_k=k)
        candidates = merge_rerank(candidates, results, top_k=k)
    else:
        candidates = candidates[:k]
    return {"similar": [_pick_fields(c) for c in candidates]}


def tool_detect_fraud_ring(coll, account_id, *, max_depth=4) -> dict:
    pipeline = build_graph_pipeline(account_id, max_depth=max_depth, collection=coll.name)
    docs = list(coll.aggregate(pipeline))
    return _jsonable(summarize_ring(docs[0] if docs else {"chain": []}, seed_account=account_id))


def tool_record_decision(
    db,
    transaction_id,
    decision,
    *,
    confidence,
    risk_factors,
    reasoning,
    reviewed_by,
    escalated=False,
    recommended_decision=None,
) -> dict:
    decision_doc = build_decision_doc(
        transaction_id,
        decision,
        confidence=confidence,
        risk_factors=risk_factors,
        reasoning=reasoning,
        reviewed_by=reviewed_by,
    )
    db["transaction_decisions"].insert_one(decision_doc)
    if escalated:
        audit = build_audit_event(
            "escalated_to_human",
            transaction_id,
            decision_id=decision_doc["decision_id"],
            severity="warning",
            event_data={"human_decision": decision, "recommended_decision": recommended_decision},
        )
    else:
        audit = build_audit_event(
            "decision_stored", transaction_id, decision_id=decision_doc["decision_id"]
        )
    db["audit_events"].insert_one(audit)
    # Advance the lifecycle status to past tense (approve -> approved) so a decided case matches
    # the seed's vocabulary and DECIDED_STATUSES, making it eligible as precedent in later reviews.
    status = {"approve": "approved", "reject": "rejected"}.get(decision, decision)
    db["transactions"].update_one({"transaction_id": transaction_id}, {"$set": {"status": status}})
    return {"recorded": True, "decision_id": decision_doc["decision_id"]}

In [13]:
TOOLS = [
    {
        "type": "custom",
        "name": "verify_mandates",
        "description": "Validate the AP2 Checkout and Payment Mandate JWTs (signature, constraints, "
        "double-spend). Run this FIRST. If valid=false, constraints_satisfied=false, or "
        "double_spend_detected=true, reject immediately.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "get_transaction",
        "description": "Fetch the full transaction record under review.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "hybrid_search_similar_frauds",
        "description": "Retrieve the most similar prior (already-decided) cases as precedent, using "
        "hybrid vector + full-text search.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "k": {"type": "integer", "description": "how many precedents (default 5)"},
            },
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "detect_fraud_ring",
        "description": "Trace the account's sender->recipient chain for circular-flow / mule / layering patterns.",
        "input_schema": {
            "type": "object",
            "properties": {"account_id": {"type": "string"}},
            "required": ["account_id"],
        },
    },
    {
        "type": "custom",
        "name": "record_decision",
        "description": "Persist the final approve/reject decision with reasoning and an audit event.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "risk_factors": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "escalated": {"type": "boolean"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
            },
            "required": ["transaction_id", "decision", "reasoning"],
        },
    },
    {
        "type": "custom",
        "name": "escalate",
        "description": "Send a risky case to a human reviewer for the final decision. Use for "
        "medium-confidence, high-value, structuring, or fraud-ring cases.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "reason": {"type": "string"},
            },
            "required": ["transaction_id", "recommended_decision", "reason"],
        },
    },
]

In [14]:
# Host-side handlers — map each data tool to its handler function.
reranker = (
    (lambda q, ds, top_k: rerank(q, ds, client=ai_client, top_k=top_k))
    if (ENABLE_RERANK and ai_client)
    else None
)


def _verify_mandates(inp):
    result = tool_verify_mandates(db, inp["transaction_id"], ts_public_key)
    ok = result["valid"] and result["constraints_satisfied"] and not result["double_spend_detected"]
    print(f"\n  [verify_mandates] {inp['transaction_id']}: {'pass' if ok else 'FAIL'}")
    return result


def _hybrid(inp):
    return tool_hybrid_search_similar_frauds(
        coll,
        inp["transaction_id"],
        inp.get("k", 5),
        enable_rerank=ENABLE_RERANK,
        reranker=reranker,
    )


def _record(inp):
    result = tool_record_decision(
        db,
        inp["transaction_id"],
        inp["decision"],
        confidence=inp.get("confidence", 0),
        risk_factors=inp.get("risk_factors", []),
        reasoning=inp.get("reasoning", ""),
        reviewed_by="human" if inp.get("escalated") else "agent",
        escalated=inp.get("escalated", False),
        recommended_decision=inp.get("recommended_decision"),
    )
    if (
        inp["decision"] == "approve"
    ):  # store the AP2 receipt that later powers double-spend detection
        txn_doc = coll.find_one(
            {"transaction_id": inp["transaction_id"]},
            {"checkout_mandate_jwt": 1, "mandate_id": 1, "agent_pk": 1},
        )
        if txn_doc and txn_doc.get("mandate_id"):
            checkout_hash = hashlib.sha256(txn_doc["checkout_mandate_jwt"].encode()).hexdigest()
            store_mandate_receipt(
                db, txn_doc["mandate_id"], txn_doc["agent_pk"], checkout_hash, "approve"
            )
    return result


HANDLERS = {
    "verify_mandates": _verify_mandates,
    "get_transaction": lambda inp: tool_get_transaction(coll, inp["transaction_id"]),
    "hybrid_search_similar_frauds": _hybrid,
    "detect_fraud_ring": lambda inp: tool_detect_fraud_ring(coll, inp["account_id"]),
    "record_decision": _record,
}

### Create the agent, environment, and session

`model`, `system`, and `tools` live on the agent. Networking is `limited` -- MongoDB is reached only through the host-side round-trip.

In [15]:
SYSTEM = """You are a financial fraud reviewer. You will be given transaction IDs to review.

For EACH transaction, in order:
1. Call get_transaction to read it.
2. Call verify_mandates to validate the AP2 mandate chain (signature, constraints, double-spend).
   HARD GATE: if verify_mandates returns valid=false, constraints_satisfied=false, OR
   double_spend_detected=true, call record_decision immediately with decision="reject" and
   skip the remaining steps for that transaction.
3. Call hybrid_search_similar_frauds to retrieve similar decided precedents.
4. Call detect_fraud_ring on the sender's account_number to check for ring/mule patterns.
5. Weigh the precedents, the ring signal, the amount, and your confidence. Then make EXACTLY
   ONE terminal call for that transaction:
   - You MUST call escalate (do NOT call record_decision yourself) whenever ANY of these holds:
       * a structuring amount ($4,900-$4,999),
       * a high-value amount (>= $50,000) that you would otherwise approve,
       * detect_fraud_ring reports suspicious_patterns, or
       * your confidence is medium (~75-85).
     Give your recommended_decision and a short reason.
   - Otherwise (a clear-cut case matching none of the above), call record_decision (approve/reject).

When you escalate, you will receive the human's decision. Then call record_decision with that
decision, escalated=true, and recommended_decision set to what you had recommended.

Be concise. Move to the next transaction after recording a decision."""

agent = client.beta.agents.create(
    name="MongoDB Atlas fraud reviewer", model=MODEL, system=SYSTEM, tools=TOOLS
)
environment = client.beta.environments.create(
    name="fraud-review-env", config={"type": "cloud", "networking": {"type": "limited"}}
)
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Fraud review",
)
print("session:", session.id)
print(f"Watch in Console: https://platform.claude.com/workspaces/default/sessions/{session.id}")

session: sesn_01NkVQkGDKzGsuwR62SfGRaq
Watch in Console: https://platform.claude.com/workspaces/default/sessions/sesn_01NkVQkGDKzGsuwR62SfGRaq


### Where to watch the run

The cell above prints a **Console link** (`platform.claude.com/workspaces/default/sessions/...`). Open it in a browser to see:

- Each `custom_tool_use` call the agent makes (tool name + input JSON)
- The session idling at `requires_action` while your notebook runs the `pymongo` handler
- The `escalate` pauses where the agent waits for a human decision
- The agent's final text output summarizing its verdicts

After the run completes, check **MongoDB Atlas** to see the results:

- **`transactions`** collection: each reviewed case now has `status: "approved"` or `"rejected"` (was `"pending"`)
- **`transaction_decisions`** collection: one immutable verdict doc per case with reasoning, confidence, and `reviewed_by` (agent vs. human)
- **`audit_events`** collection: append-only trail including `escalated_to_human` events that record both the agent's recommendation and the human's final call
- **`mandate_receipts`** collection: AP2 receipts for approved transactions (powers double-spend detection on re-runs)

### Run the review with the human gate

Data-tool calls are serviced automatically from `HANDLERS`. An `escalate` call pauses for a human -- you'll see an interactive prompt in the cell output asking `Your decision [approve/reject]:`.

With `AUTO_APPROVE=1` set, the gate resolves deterministically: it concurs with the agent on most cases but **overrides on `txn-review-struct`** (the structuring case) to demonstrate a differing human verdict.

> **Tip:** If typing in the input box doesn't work (Jupyter intercepts single keys like "o"), click directly on the input field or copy-paste your answer.

Human resolver -- handles escalation prompts (interactive or `AUTO_APPROVE`):

In [ ]:
OVERRIDE_IDS = {"txn-review-struct"}


def resolve_human_decision(recommendation, *, auto_approve, override_ids, txn_id) -> str:
    """Stand in for a human reviewer. In CI (auto_approve) it concurs with the agent, except on
    the seeded override_ids where it flips the verdict so a differing human decision is visible."""
    opposite = "reject" if recommendation == "approve" else "approve"
    if auto_approve and txn_id in set(override_ids):
        return opposite
    return recommendation


def resolver(tool_input):
    txn_id = tool_input.get("transaction_id", "")
    recommended = tool_input.get("recommended_decision", "reject")
    if AUTO_APPROVE:
        decision = resolve_human_decision(
            recommended, auto_approve=True, override_ids=OVERRIDE_IDS, txn_id=txn_id
        )
        flag = "  <-- HUMAN OVERRIDE" if decision != recommended else ""
        print(f"  [human/auto] {txn_id}: agent recommended {recommended} -> human {decision}{flag}")
        return decision
    print(
        f"\n  ESCALATED {txn_id} (agent recommends {recommended}): {tool_input.get('reason', '')}"
    )
    return (
        "approve"
        if input("  Your decision [approve/reject]: ").strip().lower().startswith("a")
        else "reject"
    )


The gate loop -- services `custom_tool_use` events from `HANDLERS` and routes `escalate` to the human resolver:

In [ ]:
def send_result(custom_tool_use_id, result):
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.custom_tool_result",
                "custom_tool_use_id": custom_tool_use_id,
                "content": [{"type": "text", "text": json.dumps(result)}],
            }
        ],
    )


def run_gate_loop(stream, send, *, handlers, resolver) -> dict:
    """Drive the session to completion, servicing custom-tool calls as they arrive.

    This is the whole Path A round-trip for a real workload. Each `agent.custom_tool_use`
    event names a tool and its input; the session then goes idle with
    `stop_reason == "requires_action"` until we answer. We look up the tool in `handlers`
    (which run `pymongo` host-side — the credential never leaves this process), and route the
    special `escalate` tool to the human `resolver` instead. We post each result back with
    `send`, and the session resumes. Break on a terminal idle (`end_turn`) or termination.
    """
    pending: dict[str, Any] = {}
    responded: set[str] = set()
    serviced: list[tuple[str, dict]] = []
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            pending[ev.id] = ev
        elif ev.type == "session.status_idle":
            stop = getattr(ev, "stop_reason", None)
            if stop is None:
                continue
            if stop.type != "requires_action":
                break  # end_turn / retries_exhausted — the run is done
            for event_id in stop.event_ids or []:
                if event_id in responded:
                    continue
                call = pending.get(event_id)
                if call is None:
                    continue
                if call.name == "escalate":
                    result = {"human_decision": resolver(call.input)}
                else:
                    handler = handlers.get(call.name)
                    result = (
                        handler(call.input) if handler else {"error": f"unknown tool {call.name}"}
                    )
                responded.add(event_id)
                send(event_id, result)
                serviced.append((call.name, result))
        elif ev.type == "session.status_terminated":
            break
    return {"serviced": serviced, "responded": sorted(responded)}


Kick off the review. The agent will process all 5 pending transactions, escalating risky cases for your input:

In [16]:
kickoff = {
    "type": "user.message",
    "content": [
        {"type": "text", "text": "Review these flagged transactions: " + ", ".join(PENDING)}
    ],
}
run_start = datetime.now(UTC)

with client.beta.sessions.events.stream(session_id=session.id) as stream:
    client.beta.sessions.events.send(session_id=session.id, events=[kickoff])
    gate_result = run_gate_loop(stream, send_result, handlers=HANDLERS, resolver=resolver)

wait_for_idle_status(client, session.id)
print(f"\nserviced {len(gate_result['serviced'])} tool calls")


  [verify_mandates] txn-review-clean: pass

  [verify_mandates] txn-review-fraud: pass

  [verify_mandates] txn-review-struct: pass

  [verify_mandates] txn-review-high: pass

  [verify_mandates] txn-review-ring: pass

  ESCALATED txn-review-struct (agent recommends reject): Cash deposit of $4,950 just below $5,000 CTR threshold, following two similar deposits earlier in the week. Structuring pattern matches all similar precedent rejections. Amount falls in structuring zone ($4,900–$4,999).


  Your decision [approve/reject]:  reject



  ESCALATED txn-review-high (agent recommends approve): High-value wire ($75,000) from legitimate business (Northwind Foods) to domestic logistics partner (Pacific Logistics) for bulk shipment. Matches approved precedent (txn-high-01: $120k approved). Escalated per policy for amounts ≥ $50,000 that would otherwise approve.


  Your decision [approve/reject]:  approve



  ESCALATED txn-review-ring (agent recommends reject): $880 transfer between Quartz Trading (ACC-RING-A) and Onyx Imports (ACC-RING-B) in active money-mule ring. Fraud ring detection shows circular flow across 3+ accounts with suspicious patterns. All similar prior transactions rejected. Escalated per ring-detection policy.


  Your decision [approve/reject]:  approve



serviced 28 tool calls


### Review the decisions and audit trail

Read decisions and audit events back from MongoDB (append-only, scoped to this run). The output shows:

- **Decisions by lane** -- which transaction type got which verdict
- **Escalation count** -- how many cases went to the human gate
- **Human overrides** -- cases where the human disagreed with the agent's recommendation

The closing asserts are a self-check: all 5 lanes decided, correct count, at least one escalation.

In [17]:
from collections import Counter

decisions = list(db["transaction_decisions"].find({"created_at": {"$gte": run_start}}, {"_id": 0}))
audits = list(db["audit_events"].find({"timestamp": {"$gte": run_start}}, {"_id": 0}))

print("Decisions by lane:")
lanes = Counter()
for d in decisions:
    txn = coll.find_one({"transaction_id": d["transaction_id"]})
    lanes[f"{(txn['lane'] if txn else '?')} -> {d['decision']}"] += 1
for k, v in sorted(lanes.items()):
    print(f"  {k}: {v}")

overrides = [
    a
    for a in audits
    if a["event_type"] == "escalated_to_human"
    and a["event_data"].get("human_decision") != a["event_data"].get("recommended_decision")
]
print(
    f"\nescalated_to_human events: {sum(1 for a in audits if a['event_type'] == 'escalated_to_human')}"
)
print(f"human overrides (verdict != agent recommendation): {len(overrides)}")
for a in overrides:
    ed = a["event_data"]
    print(
        f"  {a['transaction_id']}: agent {ed['recommended_decision']} -> human {ed['human_decision']}"
    )

# Self-check — fail loudly if this run didn't produce the expected result.
decided_lanes = {coll.find_one({"transaction_id": d["transaction_id"]})["lane"] for d in decisions}
assert decided_lanes == {"clean_approve", "clear_reject", "high_value", "ring", "structuring"}, (
    f"expected all 5 lanes decided, saw {sorted(decided_lanes)}"
)
assert len(decisions) == len(PENDING), f"expected {len(PENDING)} decisions, got {len(decisions)}"
assert sum(1 for a in audits if a["event_type"] == "escalated_to_human") >= 1, (
    "expected >=1 escalation"
)
if AUTO_APPROVE:
    assert overrides, "AUTO_APPROVE run should show the seeded override on txn-review-struct"
print("\nself-check OK")

Decisions by lane:
  clean_approve -> approve: 1
  clear_reject -> reject: 1
  high_value -> approve: 1
  ring -> approve: 1
  structuring -> reject: 1

escalated_to_human events: 3
human overrides (verdict != agent recommendation): 1
  txn-review-ring: agent reject -> human approve

self-check OK


## 4. MongoDB Atlas as the system of record and audit backbone

Step back and look at where the run's state lives: **everything the agent did is durable in the same cluster it retrieved from.** That is the single-engine payoff on the write side:

- **`transactions`** -- the operational record (each case advances `pending` -> `approved` or `rejected`)
- **`transaction_decisions`** -- immutable verdicts with reasoning, confidence, risk factors
- **`audit_events`** -- append-only trail; escalations record both agent recommendation and human call
- **`mandate_receipts`** -- AP2 receipts powering double-spend detection

No separate audit system, no event bus, no second database. One cluster is the system of record *and* the compliance surface.

```mermaid
flowchart LR
    subgraph transactions["transactions collection"]
        PENDING["status: pending"] --> APPROVED["status: approved"]
        PENDING --> REJECTED["status: rejected"]
    end

    subgraph decisions["transaction_decisions\n(append-only)"]
        DEC["decision_id\ntransaction_id\ndecision\nconfidence_score\nrisk_factors\nreasoning\nreviewed_by\ncreated_at"]
    end

    subgraph audit["audit_events\n(append-only)"]
        AE_STORED["event_type:\ndecision_stored"]
        AE_ESC["event_type:\nescalated_to_human\n(+ human vs agent verdict)"]
    end

    subgraph mandates["mandate_receipts"]
        MR["mandate_id\nagent_pk\ncheckout_hash\ndecision"]
    end

    APPROVED --> DEC
    REJECTED --> DEC
    DEC --> AE_STORED
    DEC --> AE_ESC
    APPROVED --> MR

    style transactions fill:#e8f4e8,stroke:#2d6a2d
    style decisions fill:#e8e8f4,stroke:#2d2d6a
    style audit fill:#f4f4e8,stroke:#6a6a2d
    style mandates fill:#f4e8f4,stroke:#6a2d6a
```

One case, end to end:

In [18]:
case = "txn-review-struct"
txn = coll.find_one({"transaction_id": case}, {"_id": 0, "embedding": 0})
print(f"operational record: {case}  lane={txn['lane']}  status={txn['status']}")

decision = db["transaction_decisions"].find_one(
    {"transaction_id": case, "created_at": {"$gte": run_start}}, {"_id": 0}
)
print(
    f"decision record:    {decision['decision']}  reviewed_by={decision['reviewed_by']}  confidence={decision['confidence_score']}"
)

print("audit trail (append-only):")
for a in (
    db["audit_events"]
    .find({"transaction_id": case, "timestamp": {"$gte": run_start}}, {"_id": 0})
    .sort("timestamp", 1)
):
    extra = ""
    if a["event_type"] == "escalated_to_human":
        ed = a["event_data"]
        extra = f"  (agent recommended {ed['recommended_decision']}, human decided {ed['human_decision']})"
    print(f"  {a['timestamp']:%H:%M:%S}  {a['event_type']}  severity={a['severity']}{extra}")

txn_mandate = coll.find_one({"transaction_id": case}, {"mandate_id": 1, "agent_pk": 1})
if txn_mandate and txn_mandate.get("mandate_id"):
    receipt = db["mandate_receipts"].find_one({"mandate_id": txn_mandate["mandate_id"]}, {"_id": 0})
    if receipt:
        print(
            f"mandate receipt:    mandate_id={receipt['mandate_id'][:12]}…  decision={receipt['decision']}"
        )

operational record: txn-review-struct  lane=structuring  status=rejected
decision record:    reject  reviewed_by=human  confidence=0.92
audit trail (append-only):
  16:31:37  escalated_to_human  severity=warning  (agent recommended reject, human decided reject)


**Production:** Replace the streaming loop with a webhook for `session.status_idled`. The data-tool handlers are identical; only the trigger changes. See [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb).

In [19]:
client.beta.sessions.archive(session.id)
client.beta.environments.archive(environment.id)
# Agents are reusable across runs; archiving is optional and permanent.
mongo.close()
print("archived session + environment; closed MongoDB connection")

archived session + environment; closed MongoDB connection


## 5. Cheat sheet

### Pipeline Builder Signatures

```python
build_vector_pipeline(qvec, *, limit, candidates=None, vector_index, status_in)
build_lexical_pipeline(query, *, limit, search_index)
build_rank_fusion_pipeline(qvec, query, *, k, vector_index, search_index, status_in)
build_graph_pipeline(account_id, *, max_depth=4, collection="transactions")
summarize_ring(graph_doc, *, seed_account) -> dict  # circular_flow, layering, network_size
```

### Tool Schemas

| Tool | Required Params | Returns |
|------|----------------|---------|
| `verify_mandates` | `transaction_id` | `{valid, constraints_satisfied, double_spend_detected}` |
| `get_transaction` | `transaction_id` | Full transaction document (minus embedding) |
| `hybrid_search_similar_frauds` | `transaction_id`, optional `k` (default 5) | `{similar: [{transaction_id, text, amount, decision, ...}]}` |
| `detect_fraud_ring` | `account_id` | `{network_size, circular_flow, layering, suspicious_patterns}` |
| `record_decision` | `transaction_id`, `decision`, `reasoning` | `{recorded: true, decision_id}` |
| `escalate` | `transaction_id`, `recommended_decision`, `reason` | Routed to human; returns `{human_decision}` |

### Key Config Values

| Constant | Value | Source |
|----------|-------|--------|
| Embedding dimensions | 1024 | `EMBED_DIM` |
| Vector index name | `vector_index` | `config.VECTOR_INDEX_NAME` |
| Search index name | `search_index` | `config.SEARCH_INDEX_NAME` |
| Decided statuses filter | `["approved", "rejected"]` | `config.DECIDED_STATUSES` |
| Default model | `claude-haiku-4-5` | `resolve_model()` |
| Database | `fraud_review_demo` | hardcoded |
| Collection | `transactions` | hardcoded |
| Minimum MongoDB version | 8.0+ (for `$rankFusion`) | Runtime check |

## 6. Recap

You connected MongoDB Atlas to a Claude Managed Agent -- from connection to system of record in one build:

- **Connect** -- three credential-safe data paths; the MongoDB secret always on your side.
- **Retrieve** -- vector, full-text, hybrid RRF, and graph traversal as liftable pipeline builders.
- **Gate** -- risky decisions behind CMA's native `requires_action` human pause.
- **Persist** -- decisions, append-only audit trail, and AP2 receipts in the same cluster.

Swap the collection and the tools, and the same shape grounds any agent in MongoDB.